### LOADING DATASETS - 2024 DROPED ###

In [13]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# ============================================
# STEP 1: READ ALL DATASETS & FILTER 2024
# ============================================

print("="*80)
print("STEP 1: LOADING & FILTERING DATASETS")
print("="*80)

# Define file paths
data_path = '/home/jupyter/redo/raw_data/mcphases-a-dataset-of-physiological-hormonal-and-self-reported-events-and-symptoms-for-menstrual-health-tracking-with-wearables-1.0.0/'

files = [
    'calories.csv',
    'computed_temperature.csv',
    'demographic_vo2_max.csv',
    'estimated_oxygen_variation.csv',
    'exercise.csv',
    'glucose.csv',
    'heart_rate.csv',
    'hormones_and_selfreport.csv',
    'respiratory_rate_summary.csv',
    'sleep.csv',
    'sleep_score.csv',
    'steps.csv',
    'stress_score.csv',
    'subject-info.csv',
    'time_in_heart_rate_zones.csv',
    'wrist_temperature.csv'
]

# Files that need special engine='python' handling
problematic_files = ['wrist_temperature.csv']

# Dictionary to store dataframes
datasets = {}

for file in files:
    print(f"\nLoading {file}...")
    try:
        # Try normal loading first
        if file in problematic_files:
            print(f"   Using engine='python' for {file}...")
            df = pd.read_csv(data_path + file, engine='python')
        else:
            df = pd.read_csv(data_path + file)
            
        print(f"   Initial shape: {df.shape}")
        
        # Filter out 2024 data (if study_interval column exists)
        if 'study_interval' in df.columns:
            original_rows = len(df)
            df = df[df['study_interval'] == 2022].copy()
            removed_rows = original_rows - len(df)
            print(f"   Removed {removed_rows} rows from 2024")
            print(f"   Final shape: {df.shape}")
        else:
            print(f"   No 'study_interval' column - keeping all data")
        
        # Store with clean name
        dataset_name = file.replace('.csv', '').replace('-', '_')
        datasets[dataset_name] = df
        
    except Exception as e:
        print(f"   ERROR loading {file}: {e}")
        # Try one more time with engine='python' if not already tried
        if file not in problematic_files:
            try:
                print(f"   Retrying with engine='python'...")
                df = pd.read_csv(data_path + file, engine='python')
                print(f"   SUCCESS! Initial shape: {df.shape}")
                
                if 'study_interval' in df.columns:
                    original_rows = len(df)
                    df = df[df['study_interval'] == 2022].copy()
                    removed_rows = original_rows - len(df)
                    print(f"   Removed {removed_rows} rows from 2024")
                    print(f"   Final shape: {df.shape}")
                else:
                    print(f"   No 'study_interval' column - keeping all data")
                
                dataset_name = file.replace('.csv', '').replace('-', '_')
                datasets[dataset_name] = df
                
            except Exception as e2:
                print(f"   FAILED AGAIN: {e2}")

print(f"\nSuccessfully loaded {len(datasets)} datasets")

STEP 1: LOADING & FILTERING DATASETS

Loading calories.csv...
   Initial shape: (20166975, 6)
   Removed 14846405 rows from 2024
   Final shape: (5320570, 6)

Loading computed_temperature.csv...
   Initial shape: (5575, 14)
   Removed 2317 rows from 2024
   Final shape: (3258, 14)

Loading demographic_vo2_max.csv...
   Initial shape: (11482, 8)
   Removed 7951 rows from 2024
   Final shape: (3531, 8)

Loading estimated_oxygen_variation.csv...
   Initial shape: (3070312, 6)
   Removed 1143619 rows from 2024
   Final shape: (1926693, 6)

Loading exercise.csv...
   Initial shape: (7282, 26)
   Removed 2871 rows from 2024
   Final shape: (4411, 26)

Loading glucose.csv...
   Initial shape: (837130, 6)
   Removed 0 rows from 2024
   Final shape: (837130, 6)

Loading heart_rate.csv...
   Initial shape: (63100276, 7)
   Removed 23287581 rows from 2024
   Final shape: (39812695, 7)

Loading hormones_and_selfreport.csv...
   Initial shape: (5659, 22)
   Removed 1961 rows from 2024
   Final shap

### MISSING VALUE EXAMINATION ###

In [15]:
# ============================================
# STEP 2: MISSING VALUE ANALYSIS
# ============================================

print("\n" + "="*80)
print("STEP 2: MISSING VALUE REPORT")
print("="*80)

missing_report = []

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    
    # Calculate missing percentages
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    if missing_pct.sum() > 0:
        print(f"\n   Missing values:")
        for col, pct in missing_pct[missing_pct > 0].items():
            print(f"      {col}: {pct}%")
            missing_report.append({
                'dataset': name,
                'column': col,
                'missing_pct': pct
            })
    else:
        print(f"   No missing values!")



STEP 2: MISSING VALUE REPORT

CALORIES
   Shape: (5320570, 6)
   Columns: ['id', 'study_interval', 'is_weekend', 'day_in_study', 'timestamp', 'calories']
   No missing values!

COMPUTED_TEMPERATURE
   Shape: (3258, 14)
   Columns: ['id', 'study_interval', 'is_weekend', 'sleep_start_day_in_study', 'sleep_start_timestamp', 'sleep_end_day_in_study', 'sleep_end_timestamp', 'type', 'temperature_samples', 'nightly_temperature', 'baseline_relative_sample_sum', 'baseline_relative_sample_sum_of_squares', 'baseline_relative_nightly_standard_deviation', 'baseline_relative_sample_standard_deviation']

   Missing values:
      baseline_relative_sample_sum: 2.09%
      baseline_relative_sample_sum_of_squares: 2.09%
      baseline_relative_nightly_standard_deviation: 2.09%
      baseline_relative_sample_standard_deviation: 2.09%

DEMOGRAPHIC_VO2_MAX
   Shape: (3531, 8)
   Columns: ['id', 'study_interval', 'is_weekend', 'day_in_study', 'demographic_vo2_max', 'demographic_vo2_max_error', 'filtered_dem

### AGGREGATION PREPARE ###

In [17]:
print("="*80)
print("STEP 1: DROP UNNECESSARY COLUMNS")
print("="*80)

# Drop pdg from hormones_and_selfreport (100% missing)
if 'pdg' in datasets['hormones_and_selfreport'].columns:
    datasets['hormones_and_selfreport'].drop('pdg', axis=1, inplace=True)
    print("Dropped 'pdg' from hormones_and_selfreport")

# Drop elevationgain from exercise (9.23% missing, not critical)
if 'elevationgain' in datasets['exercise'].columns:
    datasets['exercise'].drop('elevationgain', axis=1, inplace=True)
    print("Dropped 'elevationgain' from exercise")

print("\n" + "="*80)
print("STEP 2: STANDARDIZE DAY_IN_STUDY COLUMNS")
print("="*80)

# Sleep-related datasets: use sleep_end_day_in_study as day_in_study

# Computed temperature
if 'sleep_end_day_in_study' in datasets['computed_temperature'].columns:
    datasets['computed_temperature']['day_in_study'] = datasets['computed_temperature']['sleep_end_day_in_study']
    print("  computed_temperature: Added 'day_in_study' from 'sleep_end_day_in_study'")
    print(f"    Shape: {datasets['computed_temperature'].shape}")

# Sleep
if 'sleep_end_day_in_study' in datasets['sleep'].columns:
    datasets['sleep']['day_in_study'] = datasets['sleep']['sleep_end_day_in_study']
    print("  sleep: Added 'day_in_study' from 'sleep_end_day_in_study'")
    print(f"    Shape: {datasets['sleep'].shape}")

# Exercise: use start_day_in_study as day_in_study
if 'start_day_in_study' in datasets['exercise'].columns:
    datasets['exercise']['day_in_study'] = datasets['exercise']['start_day_in_study']
    print("  exercise: Added 'day_in_study' from 'start_day_in_study'")
    print(f"    Shape: {datasets['exercise'].shape}")

print("\n" + "="*80)
print("STEP 3: VERIFY ALL DATASETS NOW HAVE 'day_in_study' (except subject-level)")
print("="*80)

for name, df in datasets.items():
    has_day = 'day_in_study' in df.columns
    has_id = 'id' in df.columns
    
    if name in ['subject_info']:  # FIXED: removed 'height_and_weight'
        print(f"{name:<40} Subject-level (id only): {has_id}")
    else:
        print(f"{name:<40} Has day_in_study: {has_day}, Has id: {has_id}")

print("\n" + "="*80)
print("PREPARATION COMPLETE!")
print("="*80)

STEP 1: DROP UNNECESSARY COLUMNS
Dropped 'pdg' from hormones_and_selfreport
Dropped 'elevationgain' from exercise

STEP 2: STANDARDIZE DAY_IN_STUDY COLUMNS
  computed_temperature: Added 'day_in_study' from 'sleep_end_day_in_study'
    Shape: (3258, 15)
  sleep: Added 'day_in_study' from 'sleep_end_day_in_study'
    Shape: (4111, 19)
  exercise: Added 'day_in_study' from 'start_day_in_study'
    Shape: (4411, 26)

STEP 3: VERIFY ALL DATASETS NOW HAVE 'day_in_study' (except subject-level)
calories                                 Has day_in_study: True, Has id: True
computed_temperature                     Has day_in_study: True, Has id: True
demographic_vo2_max                      Has day_in_study: True, Has id: True
estimated_oxygen_variation               Has day_in_study: True, Has id: True
exercise                                 Has day_in_study: True, Has id: True
glucose                                  Has day_in_study: True, Has id: True
heart_rate                              

In [19]:
#print("Doing something important...")
#input("Press Enter to continue...")
#print("Continuing after you pressed Enter")


### AGGREGATE ###

In [21]:
# ============================================
# DEFINE AGGREGATION FUNCTIONS (FINAL CORRECTED VERSION)
# ============================================
import gc

def aggregate_high_frequency(df, value_cols, group_cols=['id', 'day_in_study', 'study_interval']):
    """Aggregate high-frequency data with mean, min, max, std"""
    agg_dict = {}
    
    for col in value_cols:
        if col in df.columns:
            agg_dict[col] = ['mean', 'min', 'max', 'std']
    
    if 'is_weekend' in df.columns:
        agg_dict['is_weekend'] = 'first'
    
    print(f"  Aggregating with columns: {list(agg_dict.keys())}")
    aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Flatten column names
    new_cols = []
    for col in aggregated.columns:
        if isinstance(col, tuple):
            if col[1] == 'first':
                new_cols.append(col[0])
            elif col[1]:
                new_cols.append(f"{col[0]}_{col[1]}")
            else:
                new_cols.append(col[0])
        else:
            new_cols.append(col)
    
    aggregated.columns = new_cols
    return aggregated


def aggregate_sum(df, value_cols, group_cols=['id', 'day_in_study', 'study_interval']):
    """Aggregate cumulative data with sum"""
    agg_dict = {}
    
    for col in value_cols:
        if col in df.columns:
            agg_dict[col] = 'sum'
    
    if 'is_weekend' in df.columns:
        agg_dict['is_weekend'] = 'first'
    
    print(f"  Aggregating (SUM) with columns: {list(agg_dict.keys())}")
    aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Flatten column names
    new_cols = []
    for col in aggregated.columns:
        if isinstance(col, tuple):
            new_cols.append(col[0])
        else:
            new_cols.append(col)
    
    aggregated.columns = new_cols
    return aggregated


def aggregate_mean(df, value_cols, group_cols=['id', 'day_in_study', 'study_interval']):
    """Aggregate data with mean only"""
    agg_dict = {}
    
    for col in value_cols:
        if col in df.columns:
            agg_dict[col] = 'mean'
    
    if 'is_weekend' in df.columns:
        agg_dict['is_weekend'] = 'first'
    
    print(f"  Aggregating (MEAN) with columns: {list(agg_dict.keys())}")
    aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Flatten column names
    new_cols = []
    for col in aggregated.columns:
        if isinstance(col, tuple):
            new_cols.append(col[0])
        else:
            new_cols.append(col)
    
    aggregated.columns = new_cols
    return aggregated


def aggregate_last(df, value_cols, group_cols=['id', 'day_in_study', 'study_interval']):
    """Aggregate daily cumulative scores with last value"""
    agg_dict = {}
    
    for col in value_cols:
        if col in df.columns:
            agg_dict[col] = 'last'
    
    if 'is_weekend' in df.columns:
        agg_dict['is_weekend'] = 'first'
    
    print(f"  Aggregating (LAST) with columns: {list(agg_dict.keys())}")
    aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Flatten column names
    new_cols = []
    for col in aggregated.columns:
        if isinstance(col, tuple):
            new_cols.append(col[0])
        else:
            new_cols.append(col)
    
    aggregated.columns = new_cols
    return aggregated


def aggregate_mixed(df, sum_cols, mean_cols, group_cols=['id', 'day_in_study', 'study_interval']):
    """Aggregate with mixed methods"""
    agg_dict = {}
    
    for col in sum_cols:
        if col in df.columns:
            agg_dict[col] = 'sum'
    
    for col in mean_cols:
        if col in df.columns:
            agg_dict[col] = 'mean'
    
    if 'is_weekend' in df.columns:
        agg_dict['is_weekend'] = 'first'
    
    print(f"  Aggregating (MIXED) with columns: {list(agg_dict.keys())}")
    aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Flatten column names
    new_cols = []
    for col in aggregated.columns:
        if isinstance(col, tuple):
            new_cols.append(col[0])
        else:
            new_cols.append(col)
    
    aggregated.columns = new_cols
    return aggregated

In [23]:
# ============================================
# AGGREGATE HIGH-FREQUENCY DATASETS
# ============================================

print("\n" + "="*80)
print("HIGH-FREQUENCY DATA AGGREGATION")
print("="*80)

# 1. HEART RATE (largest - do this first to fail fast)
print("\n1. Aggregating heart_rate (this will take 2-5 minutes)...")
heart_rate_agg = aggregate_high_frequency(
    datasets['heart_rate'], 
    value_cols=['bpm', 'confidence']
)
print(f"   Original: {datasets['heart_rate'].shape} -> Aggregated: {heart_rate_agg.shape}")
datasets['heart_rate_agg'] = heart_rate_agg

# 2. CALORIES (FIXED: use SUM for cumulative daily calories)
print("\n2. Aggregating calories...")
calories_agg = aggregate_sum(
    datasets['calories'], 
    value_cols=['calories']
)
print(f"   Original: {datasets['calories'].shape} -> Aggregated: {calories_agg.shape}")
datasets['calories_agg'] = calories_agg

# 3. WRIST TEMPERATURE
print("\n3. Aggregating wrist_temperature...")
wrist_temp_agg = aggregate_high_frequency(
    datasets['wrist_temperature'], 
    value_cols=['temperature_diff_from_baseline']
)
print(f"   Original: {datasets['wrist_temperature'].shape} -> Aggregated: {wrist_temp_agg.shape}")
datasets['wrist_temperature_agg'] = wrist_temp_agg

# 4. STEPS (FIXED: use SUM for cumulative daily steps)
print("\n4. Aggregating steps...")
steps_agg = aggregate_sum(
    datasets['steps'], 
    value_cols=['steps']
)
print(f"   Original: {datasets['steps'].shape} -> Aggregated: {steps_agg.shape}")
datasets['steps_agg'] = steps_agg

# 5. ESTIMATED OXYGEN VARIATION
print("\n5. Aggregating estimated_oxygen_variation...")
oxygen_agg = aggregate_high_frequency(
    datasets['estimated_oxygen_variation'], 
    value_cols=['infrared_to_red_signal_ratio']
)
print(f"   Original: {datasets['estimated_oxygen_variation'].shape} -> Aggregated: {oxygen_agg.shape}")
datasets['estimated_oxygen_variation_agg'] = oxygen_agg

# 6. GLUCOSE
print("\n6. Aggregating glucose...")
glucose_agg = aggregate_high_frequency(
    datasets['glucose'], 
    value_cols=['glucose_value']
)
print(f"   Original: {datasets['glucose'].shape} -> Aggregated: {glucose_agg.shape}")
datasets['glucose_agg'] = glucose_agg


HIGH-FREQUENCY DATA AGGREGATION

1. Aggregating heart_rate (this will take 2-5 minutes)...
  Aggregating with columns: ['bpm', 'confidence', 'is_weekend']
   Original: (39812695, 7) -> Aggregated: (3483, 12)

2. Aggregating calories...
  Aggregating (SUM) with columns: ['calories', 'is_weekend']
   Original: (5320570, 6) -> Aggregated: (3695, 5)

3. Aggregating wrist_temperature...
  Aggregating with columns: ['temperature_diff_from_baseline', 'is_weekend']
   Original: (4215949, 6) -> Aggregated: (3190, 8)

4. Aggregating steps...
  Aggregating (SUM) with columns: ['steps', 'is_weekend']
   Original: (2013749, 6) -> Aggregated: (3573, 5)

5. Aggregating estimated_oxygen_variation...
  Aggregating with columns: ['infrared_to_red_signal_ratio', 'is_weekend']
   Original: (1926693, 6) -> Aggregated: (3496, 8)

6. Aggregating glucose...
  Aggregating with columns: ['glucose_value', 'is_weekend']
   Original: (837130, 6) -> Aggregated: (3109, 8)


In [25]:
# ============================================
# AGGREGATE LOW-FREQUENCY DATASETS
# ============================================

print("\n" + "="*80)
print("LOW-FREQUENCY DATA AGGREGATION")
print("="*80)

# 7. RESPIRATORY RATE SUMMARY
print("\n7. Aggregating respiratory_rate_summary...")
respiratory_cols = [col for col in datasets['respiratory_rate_summary'].columns 
                   if col not in ['id', 'day_in_study', 'study_interval', 'is_weekend', 'timestamp']]
respiratory_agg = aggregate_mean(
    datasets['respiratory_rate_summary'], 
    value_cols=respiratory_cols
)
print(f"   Original: {datasets['respiratory_rate_summary'].shape} -> Aggregated: {respiratory_agg.shape}")
datasets['respiratory_rate_summary_agg'] = respiratory_agg

# 8. SLEEP SCORE
print("\n8. Aggregating sleep_score...")
sleep_score_cols = [col for col in datasets['sleep_score'].columns 
                   if col not in ['id', 'day_in_study', 'study_interval', 'is_weekend', 'timestamp']]
sleep_score_agg = aggregate_mean(
    datasets['sleep_score'], 
    value_cols=sleep_score_cols
)
print(f"   Original: {datasets['sleep_score'].shape} -> Aggregated: {sleep_score_agg.shape}")
datasets['sleep_score_agg'] = sleep_score_agg

# 9. STRESS SCORE (FIXED: use LAST for daily cumulative score)
print("\n9. Aggregating stress_score...")
stress_cols = [col for col in datasets['stress_score'].columns 
              if col not in ['id', 'day_in_study', 'study_interval', 'is_weekend', 'timestamp', 'status', 'calculation_failed']]
stress_score_agg = aggregate_last(
    datasets['stress_score'], 
    value_cols=stress_cols
)
print(f"   Original: {datasets['stress_score'].shape} -> Aggregated: {stress_score_agg.shape}")
datasets['stress_score_agg'] = stress_score_agg


LOW-FREQUENCY DATA AGGREGATION

7. Aggregating respiratory_rate_summary...
  Aggregating (MEAN) with columns: ['full_sleep_breathing_rate', 'full_sleep_standard_deviation', 'full_sleep_signal_to_noise', 'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation', 'deep_sleep_signal_to_noise', 'light_sleep_breathing_rate', 'light_sleep_standard_deviation', 'light_sleep_signal_to_noise', 'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation', 'rem_sleep_signal_to_noise', 'is_weekend']
   Original: (3097, 17) -> Aggregated: (2845, 16)

8. Aggregating sleep_score...
  Aggregating (MEAN) with columns: ['overall_score', 'composition_score', 'revitalization_score', 'duration_score', 'deep_sleep_in_minutes', 'resting_heart_rate', 'restlessness', 'is_weekend']
   Original: (3372, 12) -> Aggregated: (3248, 11)

9. Aggregating stress_score...
  Aggregating (LAST) with columns: ['stress_score', 'sleep_points', 'max_sleep_points', 'responsiveness_points', 'max_responsiveness_points', 'exertio

In [28]:
# ============================================
# SPECIAL CASE AGGREGATION
# ============================================

print("\n" + "="*80)
print("SPECIAL CASE AGGREGATION")
print("="*80)

# 10. COMPUTED TEMPERATURE
print("\n10. Aggregating computed_temperature...")
temp_cols = ['temperature_samples', 'nightly_temperature', 'baseline_relative_sample_sum',
             'baseline_relative_sample_sum_of_squares', 'baseline_relative_nightly_standard_deviation',
             'baseline_relative_sample_standard_deviation']
computed_temp_agg = aggregate_mean(
    datasets['computed_temperature'], 
    value_cols=temp_cols
)
print(f"   Original: {datasets['computed_temperature'].shape} -> Aggregated: {computed_temp_agg.shape}")
datasets['computed_temperature_agg'] = computed_temp_agg

# 11. SLEEP (FIXED: use MIXED - SUM for durations, MEAN for efficiency)
print("\n11. Aggregating sleep...")
sleep_sum_cols = ['duration', 'minutestofallasleep', 'minutesasleep', 'minutesawake', 
                  'minutesafterwakeup', 'timeinbed']
sleep_mean_cols = ['efficiency']

print(f"   Aggregating with SUM: {sleep_sum_cols}")
print(f"   Aggregating with MEAN: {sleep_mean_cols}")

sleep_agg = aggregate_mixed(
    datasets['sleep'], 
    sum_cols=sleep_sum_cols,
    mean_cols=sleep_mean_cols
)
print(f"   Original: {datasets['sleep'].shape} -> Aggregated: {sleep_agg.shape}")
datasets['sleep_agg'] = sleep_agg

# 12. EXERCISE (FIXED: use MIXED - SUM for durations/calories/steps, MEAN for heart rate)
print("\n12. Aggregating exercise...")
exercise_sum_cols = ['duration', 'activeduration', 'calories', 'steps']
exercise_mean_cols = ['averageheartrate']

# Filter to only numeric columns that exist
exercise_sum_valid = [col for col in exercise_sum_cols 
                      if col in datasets['exercise'].columns 
                      and pd.api.types.is_numeric_dtype(datasets['exercise'][col])]
exercise_mean_valid = [col for col in exercise_mean_cols 
                       if col in datasets['exercise'].columns 
                       and pd.api.types.is_numeric_dtype(datasets['exercise'][col])]

print(f"   Aggregating with SUM: {exercise_sum_valid}")
print(f"   Aggregating with MEAN: {exercise_mean_valid}")

exercise_agg = aggregate_mixed(
    datasets['exercise'], 
    sum_cols=exercise_sum_valid,
    mean_cols=exercise_mean_valid
)
print(f"   Original: {datasets['exercise'].shape} -> Aggregated: {exercise_agg.shape}")
datasets['exercise_agg'] = exercise_agg

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*80)
print("AGGREGATION COMPLETE!")
print("="*80)

print("\nDatasets after aggregation:")
for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name:<40} Shape: {str(df.shape):<20} Memory: {memory_mb:>10.2f} MB")

total_memory = sum([df.memory_usage(deep=True).sum() / 1024**2 for df in datasets.values()])
print(f"\n{'='*80}")
print(f"TOTAL MEMORY: {total_memory:.2f} MB ({total_memory/1024:.2f} GB)")


SPECIAL CASE AGGREGATION

10. Aggregating computed_temperature...
  Aggregating (MEAN) with columns: ['temperature_samples', 'nightly_temperature', 'baseline_relative_sample_sum', 'baseline_relative_sample_sum_of_squares', 'baseline_relative_nightly_standard_deviation', 'baseline_relative_sample_standard_deviation', 'is_weekend']
   Original: (3258, 15) -> Aggregated: (3258, 10)

11. Aggregating sleep...
   Aggregating with SUM: ['duration', 'minutestofallasleep', 'minutesasleep', 'minutesawake', 'minutesafterwakeup', 'timeinbed']
   Aggregating with MEAN: ['efficiency']
  Aggregating (MIXED) with columns: ['duration', 'minutestofallasleep', 'minutesasleep', 'minutesawake', 'minutesafterwakeup', 'timeinbed', 'efficiency', 'is_weekend']
   Original: (4111, 19) -> Aggregated: (3428, 11)

12. Aggregating exercise...
   Aggregating with SUM: ['duration', 'activeduration', 'calories', 'steps']
   Aggregating with MEAN: ['averageheartrate']
  Aggregating (MIXED) with columns: ['duration', '

### RESULT OF AGG ###

In [30]:
print("\n" + "="*80)
print("STEP 2: COLUMN DETAILS AND MISSING VALUE REPORT")
print("="*80)

for name in sorted(datasets.keys()):
    df = datasets[name]
    print(f"\n{'='*80}")
    print(f"{name.upper()}")
    print(f"{'='*80}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns and Missing Values:")
    print(f"{'Column Name':<50} {'Missing %':<15} {'Data Type'}")
    print("-" * 80)
    
    for col in df.columns:
        missing_pct = (df[col].isnull().sum() / len(df) * 100)
        dtype = str(df[col].dtype)
        print(f"{col:<50} {missing_pct:>6.2f}%        {dtype}")
    
    # Summary
    total_missing = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    overall_missing_pct = (total_missing / total_cells * 100) if total_cells > 0 else 0
    
    print("-" * 80)
    print(f"Overall missing: {overall_missing_pct:.2f}% ({total_missing}/{total_cells} cells)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)


STEP 2: COLUMN DETAILS AND MISSING VALUE REPORT

CALORIES
Shape: (5320570, 6)

Columns and Missing Values:
Column Name                                        Missing %       Data Type
--------------------------------------------------------------------------------
id                                                   0.00%        int64
study_interval                                       0.00%        int64
is_weekend                                           0.00%        bool
day_in_study                                         0.00%        int64
timestamp                                            0.00%        object
calories                                             0.00%        float64
--------------------------------------------------------------------------------
Overall missing: 0.00% (0/31923420 cells)

CALORIES_AGG
Shape: (3695, 5)

Columns and Missing Values:
Column Name                                        Missing %       Data Type
----------------------------------------

In [32]:
print("="*80)
print("COLUMNS IN ALL AGGREGATED DATASETS")
print("="*80)

# List of aggregated datasets to merge (excluding base)
agg_datasets = [
    'calories_agg',
    'steps_agg', 
    'heart_rate_agg',
    'wrist_temperature_agg',
    'estimated_oxygen_variation_agg',
    'glucose_agg',
    'respiratory_rate_summary_agg',
    'sleep_score_agg',
    'stress_score_agg',
    'computed_temperature_agg',
    'sleep_agg',
    'exercise_agg',
    'demographic_vo2_max',
    'time_in_heart_rate_zones'
]

# Base dataset
print(f"\nBASE: hormones_and_selfreport ({datasets['hormones_and_selfreport'].shape[0]} rows)")
print(f"Columns ({len(datasets['hormones_and_selfreport'].columns)}): {', '.join(datasets['hormones_and_selfreport'].columns.tolist())}")

# All other datasets
for name in agg_datasets:
    if name in datasets:
        cols = datasets[name].columns.tolist()
        print(f"\n{name} ({datasets[name].shape[0]} rows)")
        print(f"Columns ({len(cols)}): {', '.join(cols)}")

COLUMNS IN ALL AGGREGATED DATASETS

BASE: hormones_and_selfreport (3698 rows)
Columns (21): id, study_interval, is_weekend, day_in_study, phase, lh, estrogen, flow_volume, flow_color, appetite, exerciselevel, headaches, cramps, sorebreasts, fatigue, sleepissue, moodswing, stress, foodcravings, indigestion, bloating

calories_agg (3695 rows)
Columns (5): id, day_in_study, study_interval, calories, is_weekend

steps_agg (3573 rows)
Columns (5): id, day_in_study, study_interval, steps, is_weekend

heart_rate_agg (3483 rows)
Columns (12): id, day_in_study, study_interval, bpm_mean, bpm_min, bpm_max, bpm_std, confidence_mean, confidence_min, confidence_max, confidence_std, is_weekend

wrist_temperature_agg (3190 rows)
Columns (8): id, day_in_study, study_interval, temperature_diff_from_baseline_mean, temperature_diff_from_baseline_min, temperature_diff_from_baseline_max, temperature_diff_from_baseline_std, is_weekend

estimated_oxygen_variation_agg (3496 rows)
Columns (8): id, day_in_study,

### MERGE ###

In [34]:
# ============================================
# STEP 0: RE-AGGREGATE EXERCISE WITH ACTIVITYNAME
# ============================================

print("="*80)
print("RE-AGGREGATING EXERCISE TO INCLUDE ACTIVITYNAME")
print("="*80)

# We need to re-do exercise aggregation to keep activityname
# For activityname, we'll take the 'first' value (or could concatenate if multiple per day)

exercise_agg_dict = {
    'duration': 'sum',
    'calories': 'sum',
    'averageheartrate': 'mean',
    'activityname': 'first',  # Take first activity name for the day
    'is_weekend': 'first'
}

# Filter to only keep valid numeric columns
exercise_agg_with_activity = datasets['exercise'].groupby(['id', 'day_in_study', 'study_interval']).agg(exercise_agg_dict).reset_index()

# Flatten columns
new_cols = []
for col in exercise_agg_with_activity.columns:
    if isinstance(col, tuple):
        new_cols.append(col[0])
    else:
        new_cols.append(col)
exercise_agg_with_activity.columns = new_cols

print(f"Exercise re-aggregated: {exercise_agg_with_activity.shape}")
print(f"Columns: {exercise_agg_with_activity.columns.tolist()}")

# Replace old exercise_agg
datasets['exercise_agg'] = exercise_agg_with_activity

RE-AGGREGATING EXERCISE TO INCLUDE ACTIVITYNAME
Exercise re-aggregated: (540, 8)
Columns: ['id', 'day_in_study', 'study_interval', 'duration', 'calories', 'averageheartrate', 'activityname', 'is_weekend']


In [35]:
# ============================================
# STEP 1: PRE-MERGE ANALYSIS
# ============================================

print("\n" + "="*80)
print("PRE-MERGE ANALYSIS: ROW MATCH REPORT")
print("="*80)

base = datasets['hormones_and_selfreport'][['id', 'day_in_study', 'study_interval']].copy()
print(f"\nBase dataset (hormones_and_selfreport): {len(base)} rows")

datasets_to_check = {
    'calories_agg': datasets['calories_agg'],
    'steps_agg': datasets['steps_agg'],
    'heart_rate_agg': datasets['heart_rate_agg'],
    'wrist_temperature_agg': datasets['wrist_temperature_agg'],
    'glucose_agg': datasets['glucose_agg'],
    'respiratory_rate_summary_agg': datasets['respiratory_rate_summary_agg'],
    'sleep_score_agg': datasets['sleep_score_agg'],
    'computed_temperature_agg': datasets['computed_temperature_agg'],
    'sleep_agg': datasets['sleep_agg'],
    'exercise_agg': datasets['exercise_agg'],
    'time_in_heart_rate_zones': datasets['time_in_heart_rate_zones']
}

print("\n" + "-"*80)
for name, df in datasets_to_check.items():
    # Find matches
    merged_check = base.merge(
        df[['id', 'day_in_study', 'study_interval']],
        on=['id', 'day_in_study', 'study_interval'],
        how='inner',
        indicator=True
    )
    
    matches = len(merged_check)
    non_matches = len(df) - matches
    match_rate = (matches / len(df) * 100) if len(df) > 0 else 0
    
    print(f"\n{name:<40} Total: {len(df):>4} rows")
    print(f"{'':>40} Matches base: {matches:>4} ({match_rate:>5.1f}%)")
    print(f"{'':>40} No match: {non_matches:>4} (will be LOST in left join)")

print("\n" + "="*80)


PRE-MERGE ANALYSIS: ROW MATCH REPORT

Base dataset (hormones_and_selfreport): 3698 rows

--------------------------------------------------------------------------------

calories_agg                             Total: 3695 rows
                                         Matches base: 3695 (100.0%)
                                         No match:    0 (will be LOST in left join)

steps_agg                                Total: 3573 rows
                                         Matches base: 3573 (100.0%)
                                         No match:    0 (will be LOST in left join)

heart_rate_agg                           Total: 3483 rows
                                         Matches base: 3483 (100.0%)
                                         No match:    0 (will be LOST in left join)

wrist_temperature_agg                    Total: 3190 rows
                                         Matches base: 3190 (100.0%)
                                         No match:    0 (will be 

In [36]:
# ============================================
# STEP 2: PREPARE COLUMNS AND MERGE
# ============================================

print("\n" + "="*80)
print("MERGING ALL DATASETS")
print("="*80)

# Start with base dataset, drop study_interval
merged = datasets['hormones_and_selfreport'].copy()
merged = merged.drop('study_interval', axis=1)
print(f"\nStarting with base: {merged.shape}")

# Define column selections and renames
merge_configs = [
    {
        'name': 'calories_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 'calories'],
        'renames': {'calories': 'calories_daily'}
    },
    {
        'name': 'steps_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 'steps'],
        'renames': {'steps': 'steps_daily'}
    },
    {
        'name': 'heart_rate_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std'],
        'renames': {}
    },
    {
        'name': 'wrist_temperature_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'temperature_diff_from_baseline_mean', 'temperature_diff_from_baseline_min',
                    'temperature_diff_from_baseline_max', 'temperature_diff_from_baseline_std'],
        'renames': {
            'temperature_diff_from_baseline_mean': 'temp_diff_from_baseline_mean',
            'temperature_diff_from_baseline_min': 'temp_diff_from_baseline_min',
            'temperature_diff_from_baseline_max': 'temp_diff_from_baseline_max',
            'temperature_diff_from_baseline_std': 'temp_diff_from_baseline_std'
        }
    },
    {
        'name': 'glucose_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'glucose_value_mean', 'glucose_value_min', 'glucose_value_max', 'glucose_value_std'],
        'renames': {
            'glucose_value_mean': 'glucose_mean',
            'glucose_value_min': 'glucose_min',
            'glucose_value_max': 'glucose_max',
            'glucose_value_std': 'glucose_std'
        }
    },
    {
        'name': 'respiratory_rate_summary_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'full_sleep_breathing_rate', 'full_sleep_standard_deviation',
                    'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation',
                    'light_sleep_breathing_rate', 'light_sleep_standard_deviation',
                    'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation'],
        'renames': {}
    },
    {
        'name': 'sleep_score_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'overall_score', 'composition_score', 'revitalization_score', 'duration_score'],
        'renames': {}
    },
    {
        'name': 'computed_temperature_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 'nightly_temperature'],
        'renames': {}
    },
    {
        'name': 'sleep_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'duration', 'minutestofallasleep', 'minutesasleep', 'minutesawake', 
                    'minutesafterwakeup', 'timeinbed', 'efficiency'],
        'renames': {}
    },
    {
        'name': 'exercise_agg',
        'columns': ['id', 'day_in_study', 'study_interval', 'duration', 'calories', 'averageheartrate', 'activityname'],
        'renames': {
            'duration': 'duration_exercise',
            'calories': 'calories_exercise',
            'averageheartrate': 'heartrate_exercise',
            'activityname': 'activity_type'
        }
    },
    {
        'name': 'time_in_heart_rate_zones',
        'columns': ['id', 'day_in_study', 'study_interval', 
                    'in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1', 'below_default_zone_1'],
        'renames': {}
    }
]

# Merge each dataset
for config in merge_configs:
    name = config['name']
    print(f"\nMerging {name}...")
    
    # Select and rename columns
    df_to_merge = datasets[name][config['columns']].copy()
    if config['renames']:
        df_to_merge = df_to_merge.rename(columns=config['renames'])
    
    # Drop study_interval and is_weekend if they exist
    df_to_merge = df_to_merge.drop('study_interval', axis=1)
    if 'is_weekend' in df_to_merge.columns:
        df_to_merge = df_to_merge.drop('is_weekend', axis=1)
    
    # Left join
    before_shape = merged.shape
    merged = merged.merge(df_to_merge, on=['id', 'day_in_study'], how='left')
    print(f"   Before: {before_shape} -> After: {merged.shape}")

print(f"\nAfter daily-level merges: {merged.shape}")


MERGING ALL DATASETS

Starting with base: (3698, 20)

Merging calories_agg...
   Before: (3698, 20) -> After: (3698, 21)

Merging steps_agg...
   Before: (3698, 21) -> After: (3698, 22)

Merging heart_rate_agg...
   Before: (3698, 22) -> After: (3698, 26)

Merging wrist_temperature_agg...
   Before: (3698, 26) -> After: (3698, 30)

Merging glucose_agg...
   Before: (3698, 30) -> After: (3698, 34)

Merging respiratory_rate_summary_agg...
   Before: (3698, 34) -> After: (3698, 42)

Merging sleep_score_agg...
   Before: (3698, 42) -> After: (3698, 46)

Merging computed_temperature_agg...
   Before: (3698, 46) -> After: (3698, 47)

Merging sleep_agg...
   Before: (3698, 47) -> After: (3698, 54)

Merging exercise_agg...
   Before: (3698, 54) -> After: (3698, 58)

Merging time_in_heart_rate_zones...
   Before: (3698, 58) -> After: (3698, 62)

After daily-level merges: (3698, 62)


In [37]:
# ============================================
# STEP 3: MERGE SUBJECT-LEVEL DATA
# ============================================

print("\n" + "="*80)
print("MERGING SUBJECT-LEVEL DATA (subject_info)")
print("="*80)

# Subject info merges on 'id' only (no day_in_study)
subject_cols = ['id', 'birth_year', 'gender', 'ethnicity', 'education', 
                'sexually_active', 'self_report_menstrual_health_literacy', 'age_of_first_menarche']

subject_data = datasets['subject_info'][subject_cols].copy()

print(f"\nSubject info shape: {subject_data.shape}")
print(f"Unique subjects in subject_info: {subject_data['id'].nunique()}")
print(f"Unique subjects in merged data: {merged['id'].nunique()}")

# Left join on id only
before_shape = merged.shape
merged = merged.merge(subject_data, on='id', how='left')
print(f"\nBefore: {before_shape} -> After: {merged.shape}")

print(f"\nFinal merged shape: {merged.shape}")


MERGING SUBJECT-LEVEL DATA (subject_info)

Subject info shape: (42, 8)
Unique subjects in subject_info: 42
Unique subjects in merged data: 42

Before: (3698, 62) -> After: (3698, 69)

Final merged shape: (3698, 69)


In [38]:
# ============================================
# STEP 4: FILL EXERCISE COLUMNS WITH 0
# ============================================

print("\n" + "="*80)
print("FILLING EXERCISE COLUMNS WITH 0")
print("="*80)

exercise_numeric_cols = ['duration_exercise', 'calories_exercise', 'heartrate_exercise']

for col in exercise_numeric_cols:
    if col in merged.columns:
        missing_before = merged[col].isna().sum()
        merged[col] = merged[col].fillna(0)
        print(f"{col}: Filled {missing_before} missing values with 0")

# For activity_type (categorical), leave as NaN or fill with 'No Exercise'
if 'activity_type' in merged.columns:
    missing_activity = merged['activity_type'].isna().sum()
    print(f"\nactivity_type: {missing_activity} missing values (keeping as NaN for now)")
    print("   Note: NaN means no exercise that day")


FILLING EXERCISE COLUMNS WITH 0
duration_exercise: Filled 3158 missing values with 0
calories_exercise: Filled 3158 missing values with 0
heartrate_exercise: Filled 3158 missing values with 0

activity_type: 3158 missing values (keeping as NaN for now)
   Note: NaN means no exercise that day


### Merge result

In [39]:
# ============================================
# STEP 5: VERIFY FINAL DATASET
# ============================================

print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)

print(f"\nShape: {merged.shape}")
print(f"Columns: {len(merged.columns)}")

print(f"\n{'='*80}")
print("ALL COLUMN NAMES:")
print(f"{'='*80}")
for i, col in enumerate(merged.columns, 1):
    
    
    print(f"  {i:>2}. {col}")

print(f"\n{'='*80}")
print("MISSING VALUES PER COLUMN (sorted by count):")
print(f"{'='*80}")
missing_summary = merged.isnull().sum().sort_values(ascending=False)
for col, count in missing_summary[missing_summary > 0].items():
    pct = (count / len(merged)) * 100
    print(f"  {col:<45} {count:>5} ({pct:>5.1f}%)")

if missing_summary.sum() == 0:
    print("  No missing values!")

print(f"\n{'='*80}")
print("DATA TYPES:")
print(f"{'='*80}")
dtype_counts = merged.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} columns")


FINAL DATASET SUMMARY

Shape: (3698, 69)
Columns: 69

ALL COLUMN NAMES:
   1. id
   2. is_weekend
   3. day_in_study
   4. phase
   5. lh
   6. estrogen
   7. flow_volume
   8. flow_color
   9. appetite
  10. exerciselevel
  11. headaches
  12. cramps
  13. sorebreasts
  14. fatigue
  15. sleepissue
  16. moodswing
  17. stress
  18. foodcravings
  19. indigestion
  20. bloating
  21. calories_daily
  22. steps_daily
  23. bpm_mean
  24. bpm_min
  25. bpm_max
  26. bpm_std
  27. temp_diff_from_baseline_mean
  28. temp_diff_from_baseline_min
  29. temp_diff_from_baseline_max
  30. temp_diff_from_baseline_std
  31. glucose_mean
  32. glucose_min
  33. glucose_max
  34. glucose_std
  35. full_sleep_breathing_rate
  36. full_sleep_standard_deviation
  37. deep_sleep_breathing_rate
  38. deep_sleep_standard_deviation
  39. light_sleep_breathing_rate
  40. light_sleep_standard_deviation
  41. rem_sleep_breathing_rate
  42. rem_sleep_standard_deviation
  43. overall_score
  44. composition_s

In [42]:
import pandas as pd

merged = merged.sort_values(["id", "day_in_study"]).reset_index(drop=True)

def assign_cycle(sub):
    sub = sub.copy().reset_index(drop=True)
 
    is_new_cycle = (sub["phase"] == "Menstrual") & (sub["phase"].shift(1) != "Menstrual")
   
    sub["cycle"] = is_new_cycle.cumsum()
    
    return sub

merged = merged.groupby("id", group_keys=False).apply(assign_cycle)

merged = merged[merged["cycle"] > 0]

In [43]:
merged["cycle_length"] = (
    merged.groupby(["id", "cycle"])["day_in_study"]
            .transform("count")
)

In [ ]:
df_2022 = merged[merged["stud"] > 35].shape[0]

In [51]:
age = merged["age"].dropna()

age_mean = age.mean()
age_sd = age.std()

result = f"{age_mean:.1f} ({age_sd:.1f})"
print(result)

20.7 (2.6)


In [65]:
# 每个参与者只保留一条记录
unique_gender = merged.drop_duplicates(subset="id")[["id", "gender"]]

g = unique_gender["gender"].dropna()
total = len(g)

gender_stats = {
    level: f"{count} ({count/total*100:.0f})"
    for level, count in g.value_counts().items()
}

gender_stats



{'Woman': '37 (88)',
 'Non-binary': '2 (5)',
 'Gender Fluid': '1 (2)',
 'Other': '1 (2)',
 'Prefer not to say': '1 (2)'}

In [66]:
# 每个参与者只保留一条记录
unique_cycle = merged.drop_duplicates(subset="id")[["id", "cycle_length"]]

# Irregular = cycle_length > 35
n_irregular = (unique_cycle["cycle_length"] > 35).sum()

total = unique_cycle.shape[0]
pct_irregular = n_irregular / total * 100

print(f"Irregular: {n_irregular} ({pct_irregular:.1f}%)")



Irregular: 7 (16.7%)


In [67]:
total_participants = merged["id"].nunique()
print(total_participants)

42


In [63]:
merged = merged.sort_values(["id", "day_in_study"])
def days_since_last_period_complete(group):
    phases = group["phase"].to_numpy()
    n = len(group)
    days = np.full(n, np.nan)   # 先全部设成 NaN，非完整周期自然会被丢掉

    # 所有 Menstrual 的行号
    menstrual_idx = np.where(phases == "Menstrual")[0]

    # 少于 2 个 Menstrual，肯定没有完整周期，直接全 NaN
    if len(menstrual_idx) < 2:
        group["days_since_last_period"] = days
        return group

    # 把连续的 Menstrual 合并成一个个 block，得到 (start, end)
    blocks = []
    start = menstrual_idx[0]
    prev = menstrual_idx[0]
    for i in menstrual_idx[1:]:
        if i == prev + 1:
            prev = i
        else:
            blocks.append((start, prev))
            start = i
            prev = i
    blocks.append((start, prev))

    # 对每一对「相邻的两个 Menstrual block」之间的区间计数
    for (s1, e1), (s2, e2) in zip(blocks[:-1], blocks[1:]):
        counter = 0
        for j in range(e1 + 1, s2):   # e1 之后，到下一个 Menstrual 开始 s2 之前
            counter += 1
            days[j] = counter

    group["days_since_last_period"] = days
    return group

merged = merged.groupby("id", group_keys=False).apply(days_since_last_period_complete)


In [64]:
s = merged["days_since_last_period"].dropna()

median = s.median()
iqr25 = s.quantile(0.25)
iqr75 = s.quantile(0.75)

print(f"Days since last period, median (IQR): {median:.0f} [{iqr25:.0f}, {iqr75:.0f}]")


Days since last period, median (IQR): 12 [6, 19]


In [56]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3250 entries, 21 to 89
Data columns (total 72 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     3250 non-null   int64  
 1   is_weekend                             3250 non-null   bool   
 2   day_in_study                           3250 non-null   int64  
 3   phase                                  3249 non-null   object 
 4   lh                                     3040 non-null   float64
 5   estrogen                               3039 non-null   float64
 6   flow_volume                            2761 non-null   object 
 7   flow_color                             2761 non-null   object 
 8   appetite                               2888 non-null   object 
 9   exerciselevel                          2888 non-null   object 
 10  headaches                              2888 non-null   object 
 11  cramps    

In [ ]:
import pandas as pd
import numpy as np

# ====== 辅助函数 ======

def mean_sd(series):
    s = series.dropna()
    return f"{s.mean():.1f} ({s.std():.1f})"

def median_iqr(series):
    s = series.dropna()
    return f"{s.median():.0f} [{s.quantile(0.25):.0f}, {s.quantile(0.75):.0f}]"

def count_percent(series, value):
    s = series.dropna()
    total = len(s)
    n = (s == value).sum()
    pct = n / total * 100 if total > 0 else 0
    return f"{n} ({pct:.1f})"


# ====== 你需要在这里填你的真实变量名 ======
age_col = "birth_year"     # 或者你真实的 age 列
gender_col = "gender"
cycle_reg_col = 'cycle_length'       # 如果你有类似 "cycle_regular"，就写成字符串
hormone_col = None         # 如果你有激素避孕列，例如 "hormonal_contraception"
days_since_period_col = None   # 如果你有经期的天数变量


# ====== 生成 Table 1 ======

table1 = []

# ---------------------
# Demographics
# ---------------------
table1.append(["Demographics of study participants", ""])

# 年龄
if age_col in df.columns:
    if age_col == "birth_year":
        age = 2024 - df[age_col]
        table1.append(["Age (years), mean (SD)", mean_sd(age)])
    else:
        table1.append(["Age (years), mean (SD)", mean_sd(df[age_col])])

# Gender
if gender_col in df.columns:
    table1.append(["Gender, n (%)", ""])
    for g in df[gender_col].dropna().unique():
        table1.append([f"    {g}", count_percent(df[gender_col], g)])

# ---------------------
# Cycle Regularity
# ---------------------
if cycle_reg_col and (cycle_reg_col in df.columns):
    table1.append(["Cycle regularity, n (%)", ""])
    for v in df[cycle_reg_col].dropna().unique():
        table1.append([f"    {v}", count_percent(df[cycle_reg_col], v)])

# ---------------------
# Hormonal Contraception
# ---------------------
if hormone_col and (hormone_col in df.columns):
    table1.append(["Hormonal contraception, n (%)", ""])
    table1.append(["    Yes", count_percent(df[hormone_col], 1)])  # 假设 1 表示 yes

# ---------------------
# Days since last period
# ---------------------
if days_since_period_col and (days_since_period_col in df.columns):
    table1.append([
        "Days since last period, median (IQR)",
        median_iqr(df[days_since_period_col])
    ])


# ====== 输出结果 ======
table1_df = pd.DataFrame(table1, columns=["Variable", "Value"])
table1_df


### MISSING BY SUBJECTS CHECK ###

In [17]:
# activity_type: Fill NaN with 'No Exercise'
merged['activity_type'] = merged['activity_type'].fillna('No Exercise')

In [18]:
# ============================================
# MISSING DATA ANALYSIS BY SUBJECT AND COLUMN
# ============================================

print("="*80)
print("SUBJECT-LEVEL MISSING DATA ANALYSIS")
print("="*80)

# Define column groups for analysis
column_groups = {
    'Hormones': ['lh', 'estrogen'],
    'Glucose': ['glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std'],
    'Heart Rate': ['bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std'],
    'Temperature (Wrist)': ['temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min', 
                             'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std'],
    'Temperature (Nightly)': ['nightly_temperature'],
    'Respiratory Rate': ['full_sleep_breathing_rate', 'full_sleep_standard_deviation',
                         'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation',
                         'light_sleep_breathing_rate', 'light_sleep_standard_deviation',
                         'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation'],
    'Sleep Scores': ['overall_score', 'composition_score', 'revitalization_score', 'duration_score'],
    'Sleep Metrics': ['duration', 'minutestofallasleep', 'minutesasleep', 'minutesawake', 
                      'minutesafterwakeup', 'timeinbed', 'efficiency'],
    'Activity': ['calories_daily', 'steps_daily'],
    'Heart Rate Zones': ['in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1', 'below_default_zone_1'],
    'Self-reported Symptoms': ['appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 
                               'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 
                               'indigestion', 'bloating'],
    'Menstrual': ['flow_volume', 'flow_color']
}

# Calculate missing percentage per subject per column
print("\n" + "="*80)
print("SUBJECTS WITH >50% MISSING IN KEY COLUMNS")
print("="*80)

problematic_subjects = {}

for group_name, cols in column_groups.items():
    print(f"\n{group_name}:")
    print("-" * 80)
    
    group_issues = []
    
    for col in cols:
        if col in merged.columns:
            # Calculate missing % per subject for this column
            missing_by_subject = merged.groupby('id')[col].apply(lambda x: x.isna().sum() / len(x) * 100)
            
            # Find subjects with >50% missing
            high_missing = missing_by_subject[missing_by_subject > 50].sort_values(ascending=False)
            
            if len(high_missing) > 0:
                print(f"  {col}:")
                print(f"    Subjects with >50% missing: {len(high_missing)}/{merged['id'].nunique()}")
                print(f"    Worst cases:")
                for subject_id, pct in high_missing.head(5).items():
                    print(f"      Subject {subject_id}: {pct:.1f}% missing")
                
                # Store for summary
                for subject_id, pct in high_missing.items():
                    if subject_id not in problematic_subjects:
                        problematic_subjects[subject_id] = {}
                    problematic_subjects[subject_id][col] = pct
            else:
                print(f"  {col}: All subjects have <50% missing ✓")

SUBJECT-LEVEL MISSING DATA ANALYSIS

SUBJECTS WITH >50% MISSING IN KEY COLUMNS

Hormones:
--------------------------------------------------------------------------------
  lh: All subjects have <50% missing ✓
  estrogen: All subjects have <50% missing ✓

Glucose:
--------------------------------------------------------------------------------
  glucose_mean:
    Subjects with >50% missing: 4/42
    Worst cases:
      Subject 46: 93.3% missing
      Subject 29: 60.0% missing
      Subject 23: 51.1% missing
      Subject 32: 51.1% missing
  glucose_min:
    Subjects with >50% missing: 4/42
    Worst cases:
      Subject 46: 93.3% missing
      Subject 29: 60.0% missing
      Subject 23: 51.1% missing
      Subject 32: 51.1% missing
  glucose_max:
    Subjects with >50% missing: 4/42
    Worst cases:
      Subject 46: 93.3% missing
      Subject 29: 60.0% missing
      Subject 23: 51.1% missing
      Subject 32: 51.1% missing
  glucose_std:
    Subjects with >50% missing: 4/42
    Worst 

In [19]:
# ============================================
# SUMMARY: PROBLEMATIC SUBJECTS
# ============================================

print("\n" + "="*80)
print("SUMMARY: SUBJECTS WITH MULTIPLE HIGH-MISSING COLUMNS")
print("="*80)

if problematic_subjects:
    # Count how many columns each subject is missing >50%
    subjects_with_counts = [(subj, len(cols)) for subj, cols in problematic_subjects.items()]
    subjects_with_counts.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTotal subjects flagged: {len(problematic_subjects)}")
    print(f"\nTop subjects by number of problematic columns:")
    print("-" * 80)
    
    for subject_id, num_cols in subjects_with_counts[:10]:
        print(f"\nSubject {subject_id}: {num_cols} columns with >50% missing")
        print("  Columns:")
        for col, pct in sorted(problematic_subjects[subject_id].items(), key=lambda x: x[1], reverse=True):
            print(f"    - {col}: {pct:.1f}% missing")
else:
    print("\nNo subjects have >50% missing in any column! ✓")


SUMMARY: SUBJECTS WITH MULTIPLE HIGH-MISSING COLUMNS

Total subjects flagged: 11

Top subjects by number of problematic columns:
--------------------------------------------------------------------------------

Subject 29: 37 columns with >50% missing
  Columns:
    - temp_diff_from_baseline_mean: 66.7% missing
    - temp_diff_from_baseline_min: 66.7% missing
    - temp_diff_from_baseline_max: 66.7% missing
    - temp_diff_from_baseline_std: 66.7% missing
    - nightly_temperature: 63.3% missing
    - full_sleep_breathing_rate: 61.7% missing
    - full_sleep_standard_deviation: 61.7% missing
    - deep_sleep_breathing_rate: 61.7% missing
    - deep_sleep_standard_deviation: 61.7% missing
    - light_sleep_breathing_rate: 61.7% missing
    - light_sleep_standard_deviation: 61.7% missing
    - rem_sleep_breathing_rate: 61.7% missing
    - rem_sleep_standard_deviation: 61.7% missing
    - overall_score: 61.7% missing
    - composition_score: 61.7% missing
    - revitalization_score: 61.7

In [20]:
# ============================================
# ANALYZE CONSECUTIVE MISSING GAPS
# ============================================

print("="*80)
print("GAP ANALYSIS: CONSECUTIVE MISSING DAYS PER SUBJECT")
print("="*80)

# Focus on columns that will be forward/backward filled
ffill_cols = [
    'lh', 'estrogen',
    'glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std',
    'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std',
    'temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min', 
    'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std',
    'nightly_temperature',
    'full_sleep_breathing_rate', 'full_sleep_standard_deviation',
    'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation',
    'light_sleep_breathing_rate', 'light_sleep_standard_deviation',
    'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation',
    'overall_score', 'composition_score', 'revitalization_score', 'duration_score'
]

def get_max_consecutive_nan(series):
    """Calculate maximum consecutive NaN values in a series"""
    is_nan = series.isna()
    # Find consecutive NaN groups
    consecutive_groups = is_nan.ne(is_nan.shift()).cumsum()
    # Count max consecutive NaN
    max_consecutive = is_nan.groupby(consecutive_groups).sum().max()
    return max_consecutive if pd.notna(max_consecutive) else 0

# Analyze each column
print("\nColumns with subjects having long gaps (>14 consecutive days missing):")
print("="*80)

problematic_gaps = {}

for col in ffill_cols:
    if col not in merged.columns:
        continue
    
    print(f"\n{col}:")
    print("-" * 80)
    
    # Calculate max consecutive missing per subject
    max_gaps = merged.groupby('id')[col].apply(get_max_consecutive_nan)
    
    # Find subjects with gaps >14 days
    long_gaps = max_gaps[max_gaps > 14].sort_values(ascending=False)
    
    if len(long_gaps) > 0:
        print(f"  Subjects with >14 day gaps: {len(long_gaps)}/{merged['id'].nunique()}")
        print(f"  Longest gaps (top 5):")
        for subject_id, gap_days in long_gaps.head(5).items():
            total_days = len(merged[merged['id'] == subject_id])
            pct_missing = (merged[merged['id'] == subject_id][col].isna().sum() / total_days * 100)
            print(f"    Subject {subject_id}: {int(gap_days)} consecutive days ({pct_missing:.1f}% total missing, {total_days} total days)")
            
            if subject_id not in problematic_gaps:
                problematic_gaps[subject_id] = {}
            problematic_gaps[subject_id][col] = int(gap_days)
    else:
        print(f"  All subjects have gaps ≤14 days ✓")
    
    # Show gap distribution
    print(f"\n  Gap size distribution:")
    gaps_7_plus = len(max_gaps[max_gaps > 7])
    gaps_14_plus = len(max_gaps[max_gaps > 14])
    gaps_21_plus = len(max_gaps[max_gaps > 21])
    gaps_28_plus = len(max_gaps[max_gaps > 28])
    
    print(f"    >7 days:  {gaps_7_plus} subjects")
    print(f"    >14 days: {gaps_14_plus} subjects")
    print(f"    >21 days: {gaps_21_plus} subjects")
    print(f"    >28 days (full cycle): {gaps_28_plus} subjects")

GAP ANALYSIS: CONSECUTIVE MISSING DAYS PER SUBJECT

Columns with subjects having long gaps (>14 consecutive days missing):

lh:
--------------------------------------------------------------------------------
  Subjects with >14 day gaps: 1/42
  Longest gaps (top 5):
    Subject 8: 20 consecutive days (32.2% total missing, 90 total days)

  Gap size distribution:
    >7 days:  2 subjects
    >14 days: 1 subjects
    >21 days: 0 subjects
    >28 days (full cycle): 0 subjects

estrogen:
--------------------------------------------------------------------------------
  Subjects with >14 day gaps: 1/42
  Longest gaps (top 5):
    Subject 8: 20 consecutive days (32.2% total missing, 90 total days)

  Gap size distribution:
    >7 days:  2 subjects
    >14 days: 1 subjects
    >21 days: 0 subjects
    >28 days (full cycle): 0 subjects

glucose_mean:
--------------------------------------------------------------------------------
  Subjects with >14 day gaps: 9/42
  Longest gaps (top 5):
    

In [21]:
# ============================================
# SUBJECTS WITH MULTIPLE LONG GAPS
# ============================================

print("\n" + "="*80)
print("SUBJECTS WITH LONG GAPS (>14 days) IN MULTIPLE CRITICAL COLUMNS")
print("="*80)

if problematic_gaps:
    # Count columns with long gaps per subject
    gap_counts = [(subj, len(cols)) for subj, cols in problematic_gaps.items()]
    gap_counts.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTotal subjects with at least one long gap: {len(problematic_gaps)}")
    print(f"\nSubjects by number of columns with long gaps:")
    print("-" * 80)
    
    for subject_id, num_cols in gap_counts[:15]:
        total_days = len(merged[merged['id'] == subject_id])
        print(f"\nSubject {subject_id} ({total_days} total days): {num_cols} columns with >14 day gaps")
        print("  Longest gaps by column:")
        sorted_gaps = sorted(problematic_gaps[subject_id].items(), key=lambda x: x[1], reverse=True)
        for col, gap_days in sorted_gaps[:5]:  # Show top 5 worst gaps
            print(f"    {col}: {gap_days} consecutive days")
else:
    print("\nNo subjects have gaps >14 days in any column! ✓")


SUBJECTS WITH LONG GAPS (>14 days) IN MULTIPLE CRITICAL COLUMNS

Total subjects with at least one long gap: 13

Subjects by number of columns with long gaps:
--------------------------------------------------------------------------------

Subject 23 (90 total days): 17 columns with >14 day gaps
  Longest gaps by column:
    glucose_mean: 22 consecutive days
    glucose_min: 22 consecutive days
    glucose_max: 22 consecutive days
    glucose_std: 22 consecutive days
    overall_score: 21 consecutive days

Subject 8 (90 total days): 15 columns with >14 day gaps
  Longest gaps by column:
    lh: 20 consecutive days
    estrogen: 20 consecutive days
    temp_diff_from_baseline_mean: 19 consecutive days
    temp_diff_from_baseline_min: 19 consecutive days
    temp_diff_from_baseline_max: 19 consecutive days

Subject 29 (60 total days): 13 columns with >14 day gaps
  Longest gaps by column:
    temp_diff_from_baseline_mean: 32 consecutive days
    temp_diff_from_baseline_min: 32 consecuti

### SUBJECTS REMOVAL ###

In [22]:
# ============================================
# REMOVE PROBLEMATIC SUBJECTS AND COLUMNS
# ============================================

print("="*80)
print("REMOVING PROBLEMATIC SUBJECTS AND COLUMNS")
print("="*80)

# Subjects to exclude (9 total)
subjects_to_exclude = [1, 3, 10, 20, 27, 29, 30, 46, 49]

print(f"\nBefore exclusion:")
print(f"  Total rows: {len(merged)}")
print(f"  Total subjects: {merged['id'].nunique()}")
print(f"  Total columns: {len(merged.columns)}")

# Remove subjects
merged_clean = merged[~merged['id'].isin(subjects_to_exclude)].copy()

print(f"\nExcluded {len(subjects_to_exclude)} subjects: {subjects_to_exclude}")
print(f"  Rows removed: {len(merged) - len(merged_clean)}")

# Drop respiratory rate columns (all 8)
respiratory_cols = [
    'full_sleep_breathing_rate', 'full_sleep_standard_deviation',
    'deep_sleep_breathing_rate', 'deep_sleep_standard_deviation',
    'light_sleep_breathing_rate', 'light_sleep_standard_deviation',
    'rem_sleep_breathing_rate', 'rem_sleep_standard_deviation'
]

cols_dropped = [col for col in respiratory_cols if col in merged_clean.columns]
merged_clean = merged_clean.drop(columns=cols_dropped)

print(f"\nDropped {len(cols_dropped)} respiratory rate columns")

print(f"\nAfter exclusion:")
print(f"  Total rows: {len(merged_clean)}")
print(f"  Total subjects: {merged_clean['id'].nunique()}")
print(f"  Total columns: {len(merged_clean.columns)}")
print(f"  Average rows per subject: {len(merged_clean) / merged_clean['id'].nunique():.1f} days")

# Replace the merged dataset
merged = merged_clean.copy()
del merged_clean

print("\n" + "="*80)
print("EXCLUSION COMPLETE - Dataset cleaned")
print("="*80)

REMOVING PROBLEMATIC SUBJECTS AND COLUMNS

Before exclusion:
  Total rows: 3698
  Total subjects: 42
  Total columns: 69

Excluded 9 subjects: [1, 3, 10, 20, 27, 29, 30, 46, 49]
  Rows removed: 728

Dropped 8 respiratory rate columns

After exclusion:
  Total rows: 2970
  Total subjects: 33
  Total columns: 61
  Average rows per subject: 90.0 days

EXCLUSION COMPLETE - Dataset cleaned


In [23]:
# ============================================
# VERIFY MISSING DATA AFTER EXCLUSIONS
# ============================================

print("\n" + "="*80)
print("MISSING VALUES AFTER EXCLUSIONS")
print("="*80)

missing_summary = merged.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(merged) * 100).round(1)

print(f"\nColumns with missing data (sorted by count):")
print("-" * 80)
for col in missing_summary[missing_summary > 0].index:
    count = missing_summary[col]
    pct = missing_pct[col]
    print(f"{col:<45} {count:>5} ({pct:>5.1f}%)")

if missing_summary.sum() == 0:
    print("\nNo missing values!")

print(f"\nTotal missing cells: {missing_summary.sum():,}")
print(f"Total cells: {merged.shape[0] * merged.shape[1]:,}")
print(f"Overall missing: {(missing_summary.sum() / (merged.shape[0] * merged.shape[1]) * 100):.2f}%")


MISSING VALUES AFTER EXCLUSIONS

Columns with missing data (sorted by count):
--------------------------------------------------------------------------------
flow_volume                                     392 ( 13.2%)
flow_color                                      392 ( 13.2%)
glucose_mean                                    359 ( 12.1%)
glucose_std                                     359 ( 12.1%)
glucose_max                                     359 ( 12.1%)
glucose_min                                     359 ( 12.1%)
temp_diff_from_baseline_min                     356 ( 12.0%)
temp_diff_from_baseline_std                     356 ( 12.0%)
temp_diff_from_baseline_max                     356 ( 12.0%)
temp_diff_from_baseline_mean                    356 ( 12.0%)
nightly_temperature                             303 ( 10.2%)
overall_score                                   302 ( 10.2%)
composition_score                               302 ( 10.2%)
revitalization_score                           

In [24]:
# ============================================
# CHECK REMAINING GAPS
# ============================================

print("\n" + "="*80)
print("GAP ANALYSIS AFTER EXCLUSIONS")
print("="*80)

# Focus on critical columns
critical_cols = [
    'glucose_mean', 'bpm_mean', 'temp_diff_from_baseline_mean', 
    'nightly_temperature', 'overall_score'
]

def get_max_consecutive_nan(series):
    """Calculate maximum consecutive NaN values in a series"""
    is_nan = series.isna()
    consecutive_groups = is_nan.ne(is_nan.shift()).cumsum()
    max_consecutive = is_nan.groupby(consecutive_groups).sum().max()
    return max_consecutive if pd.notna(max_consecutive) else 0

print("\nMaximum consecutive missing days in critical columns:")
print("-" * 80)

for col in critical_cols:
    if col in merged.columns:
        max_gaps = merged.groupby('id')[col].apply(get_max_consecutive_nan)
        
        print(f"\n{col}:")
        print(f"  Max gap across all subjects: {int(max_gaps.max())} days")
        print(f"  Mean gap: {max_gaps.mean():.1f} days")
        print(f"  Subjects with gaps >14 days: {len(max_gaps[max_gaps > 14])}")
        print(f"  Subjects with gaps >28 days: {len(max_gaps[max_gaps > 28])}")
        
        if len(max_gaps[max_gaps > 14]) > 0:
            print(f"  Top 3 longest gaps:")
            for subject_id, gap in max_gaps.nlargest(3).items():
                print(f"    Subject {subject_id}: {int(gap)} days")

print("\n" + "="*80)
print("Remaining dataset is much cleaner!")
print("="*80)


GAP ANALYSIS AFTER EXCLUSIONS

Maximum consecutive missing days in critical columns:
--------------------------------------------------------------------------------

glucose_mean:
  Max gap across all subjects: 39 days
  Mean gap: 7.2 days
  Subjects with gaps >14 days: 5
  Subjects with gaps >28 days: 2
  Top 3 longest gaps:
    Subject 44: 39 days
    Subject 14: 37 days
    Subject 23: 22 days

bpm_mean:
  Max gap across all subjects: 20 days
  Mean gap: 1.8 days
  Subjects with gaps >14 days: 2
  Subjects with gaps >28 days: 0
  Top 3 longest gaps:
    Subject 23: 20 days
    Subject 8: 18 days
    Subject 4: 6 days

temp_diff_from_baseline_mean:
  Max gap across all subjects: 20 days
  Mean gap: 4.5 days
  Subjects with gaps >14 days: 3
  Subjects with gaps >28 days: 0
  Top 3 longest gaps:
    Subject 23: 20 days
    Subject 8: 19 days
    Subject 13: 16 days

nightly_temperature:
  Max gap across all subjects: 20 days
  Mean gap: 4.0 days
  Subjects with gaps >14 days: 2
  Sub

### IMPUTATION ###

In [26]:
# ============================================
# IMPUTATION STRATEGY - COMPLETE PIPELINE
# ============================================

print("="*80)
print("DATA IMPUTATION - STARTING")
print("="*80)

# Store original for comparison
merged_before_imputation = merged.copy()

print(f"\nDataset before imputation:")
print(f"  Shape: {merged.shape}")
print(f"  Total missing: {merged.isnull().sum().sum()}")
print(f"  Missing %: {(merged.isnull().sum().sum() / (merged.shape[0] * merged.shape[1]) * 100):.2f}%")

DATA IMPUTATION - STARTING

Dataset before imputation:
  Shape: (2970, 61)
  Total missing: 10735
  Missing %: 5.93%


In [27]:
# ============================================
# STEP 1: CATEGORICAL FILLS
# ============================================

print("\n" + "="*80)
print("STEP 1: FILLING CATEGORICAL COLUMNS")
print("="*80)

# A. activity_type: Fill with 'No Exercise'
if 'activity_type' in merged.columns:
    before = merged['activity_type'].isna().sum()
    merged['activity_type'] = merged['activity_type'].fillna('No Exercise')
    print(f"\nactivity_type: Filled {before} missing → 'No Exercise'")

# B. flow_volume/flow_color: Fill with 'None' (no menstruation)
if 'flow_volume' in merged.columns:
    before = merged['flow_volume'].isna().sum()
    merged['flow_volume'] = merged['flow_volume'].fillna('None')
    print(f"flow_volume: Filled {before} missing → 'None'")

if 'flow_color' in merged.columns:
    before = merged['flow_color'].isna().sum()
    merged['flow_color'] = merged['flow_color'].fillna('None')
    print(f"flow_color: Filled {before} missing → 'None'")

print(f"\nCategorical fills complete!")


STEP 1: FILLING CATEGORICAL COLUMNS

activity_type: Filled 0 missing → 'No Exercise'
flow_volume: Filled 392 missing → 'None'
flow_color: Filled 392 missing → 'None'

Categorical fills complete!


In [28]:
# ============================================
# STEP 2: FORWARD/BACKWARD FILL BY SUBJECT
# ============================================

print("\n" + "="*80)
print("STEP 2: FORWARD/BACKWARD FILL (gradual changes)")
print("="*80)

# Columns that change gradually across cycle - use ffill/bfill
ffill_cols = [
    'lh', 'estrogen',  # Hormones
    'glucose_mean', 'glucose_min', 'glucose_max', 'glucose_std',  # Glucose
    'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std',  # Heart rate
    'temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min', 
    'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std',  # Wrist temp
    'nightly_temperature',  # Nightly temp
    'overall_score', 'composition_score', 'revitalization_score', 'duration_score'  # Sleep scores
]

print("\nFilling columns with forward/backward fill by subject:")
for col in ffill_cols:
    if col in merged.columns:
        before = merged[col].isna().sum()
        # Forward fill within subject, then backward fill
        merged[col] = merged.groupby('id')[col].ffill().bfill()
        after = merged[col].isna().sum()
        filled = before - after
        print(f"  {col:<45} Filled: {filled:>4} ({before:>4} → {after:>4})")

print(f"\nForward/backward fill complete!")


STEP 2: FORWARD/BACKWARD FILL (gradual changes)

Filling columns with forward/backward fill by subject:
  lh                                            Filled:  173 ( 173 →    0)
  estrogen                                      Filled:  174 ( 174 →    0)
  glucose_mean                                  Filled:  359 ( 359 →    0)
  glucose_min                                   Filled:  359 ( 359 →    0)
  glucose_max                                   Filled:  359 ( 359 →    0)
  glucose_std                                   Filled:  359 ( 359 →    0)
  bpm_mean                                      Filled:   75 (  75 →    0)
  bpm_min                                       Filled:   75 (  75 →    0)
  bpm_max                                       Filled:   75 (  75 →    0)
  bpm_std                                       Filled:   75 (  75 →    0)
  temp_diff_from_baseline_mean                  Filled:  356 ( 356 →    0)
  temp_diff_from_baseline_min                   Filled:  356 ( 356 →  

In [29]:
# ============================================
# STEP 3: FILL SLEEP METRICS
# ============================================

print("\n" + "="*80)
print("STEP 3: FILLING SLEEP METRICS")
print("="*80)

# Sleep durations: Forward fill within subject (sleep patterns persist)
sleep_duration_cols = ['duration', 'minutestofallasleep', 'minutesasleep', 
                       'minutesawake', 'minutesafterwakeup', 'timeinbed']

print("\nSleep durations (forward fill by subject):")
for col in sleep_duration_cols:
    if col in merged.columns:
        before = merged[col].isna().sum()
        merged[col] = merged.groupby('id')[col].ffill().bfill()
        after = merged[col].isna().sum()
        filled = before - after
        print(f"  {col:<45} Filled: {filled:>4} ({before:>4} → {after:>4})")

# Efficiency: Fill with subject median (percentage metric)
if 'efficiency' in merged.columns:
    before = merged['efficiency'].isna().sum()
    merged['efficiency'] = merged.groupby('id')['efficiency'].transform(
        lambda x: x.fillna(x.median())
    )
    after = merged['efficiency'].isna().sum()
    print(f"\n  efficiency (median by subject):")
    print(f"    Filled: {before - after:>4} ({before:>4} → {after:>4})")

print(f"\nSleep metrics fill complete!")


STEP 3: FILLING SLEEP METRICS

Sleep durations (forward fill by subject):
  duration                                      Filled:  164 ( 164 →    0)
  minutestofallasleep                           Filled:  164 ( 164 →    0)
  minutesasleep                                 Filled:  164 ( 164 →    0)
  minutesawake                                  Filled:  164 ( 164 →    0)
  minutesafterwakeup                            Filled:  164 ( 164 →    0)
  timeinbed                                     Filled:  164 ( 164 →    0)

  efficiency (median by subject):
    Filled:  164 ( 164 →    0)

Sleep metrics fill complete!


In [30]:
# ============================================
# STEP 4: FILL ACTIVITY METRICS
# ============================================

print("\n" + "="*80)
print("STEP 4: FILLING ACTIVITY METRICS")
print("="*80)

# Steps: Fill with 0 (no steps = no activity)
if 'steps_daily' in merged.columns:
    before = merged['steps_daily'].isna().sum()
    merged['steps_daily'] = merged['steps_daily'].fillna(0)
    print(f"\nsteps_daily: Filled {before} missing → 0")

# Calories: Fill with subject median (baseline calorie burn)
if 'calories_daily' in merged.columns:
    before = merged['calories_daily'].isna().sum()
    merged['calories_daily'] = merged.groupby('id')['calories_daily'].transform(
        lambda x: x.fillna(x.median())
    )
    after = merged['calories_daily'].isna().sum()
    print(f"calories_daily (median by subject): Filled {before - after} ({before} → {after})")

# Heart rate zones: Fill with 0 (no data = no time in zones)
zone_cols = ['in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1', 'below_default_zone_1']
print(f"\nHeart rate zones (fill with 0):")
for col in zone_cols:
    if col in merged.columns:
        before = merged[col].isna().sum()
        merged[col] = merged[col].fillna(0)
        print(f"  {col:<45} Filled: {before:>4} → 0")

print(f"\nActivity metrics fill complete!")


STEP 4: FILLING ACTIVITY METRICS

steps_daily: Filled 73 missing → 0
calories_daily (median by subject): Filled 3 (3 → 0)

Heart rate zones (fill with 0):
  in_default_zone_3                             Filled:   72 → 0
  in_default_zone_2                             Filled:   72 → 0
  in_default_zone_1                             Filled:   72 → 0
  below_default_zone_1                          Filled:   72 → 0

Activity metrics fill complete!


In [31]:
# ============================================
# STEP 5: FILL SELF-REPORTED SYMPTOMS
# ============================================

print("\n" + "="*80)
print("STEP 5: FILLING SELF-REPORTED SYMPTOMS")
print("="*80)

# Symptoms: Forward fill within subject (persist until reported otherwise), then fill with 0
symptom_cols = ['appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 
                'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 
                'indigestion', 'bloating']

print("\nSymptoms (forward fill, then 0):")
for col in symptom_cols:
    if col in merged.columns:
        before = merged[col].isna().sum()
        # Forward fill within subject
        merged[col] = merged.groupby('id')[col].ffill()
        # Fill remaining with 0 (no symptom)
        merged[col] = merged[col].fillna(0)
        after = merged[col].isna().sum()
        filled = before - after
        print(f"  {col:<45} Filled: {filled:>4} ({before:>4} → {after:>4})")

print(f"\nSymptom fill complete!")


STEP 5: FILLING SELF-REPORTED SYMPTOMS

Symptoms (forward fill, then 0):
  appetite                                      Filled:  285 ( 285 →    0)
  exerciselevel                                 Filled:  285 ( 285 →    0)
  headaches                                     Filled:  285 ( 285 →    0)
  cramps                                        Filled:  285 ( 285 →    0)
  sorebreasts                                   Filled:  285 ( 285 →    0)
  fatigue                                       Filled:  285 ( 285 →    0)
  sleepissue                                    Filled:  285 ( 285 →    0)
  moodswing                                     Filled:  285 ( 285 →    0)
  stress                                        Filled:  285 ( 285 →    0)
  foodcravings                                  Filled:  285 ( 285 →    0)
  indigestion                                   Filled:  285 ( 285 →    0)
  bloating                                      Filled:  285 ( 285 →    0)

Symptom fill complete!


In [32]:
# ============================================
# STEP 8: HANDLE ANY REMAINING NULLS
# ============================================

print("\n" + "="*80)
print("STEP 8: HANDLING ANY REMAINING NULLS")
print("="*80)

remaining_nulls = merged.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if len(remaining_nulls) > 0:
    print(f"\nRemaining nulls found in {len(remaining_nulls)} columns:")
    for col, count in remaining_nulls.items():
        print(f"  {col}: {count} nulls")
    
    print("\nFilling remaining nulls with column median (last resort)...")
    for col in remaining_nulls.index:
        if merged[col].dtype in ['float64', 'int64']:
            median_val = merged[col].median()
            merged[col] = merged[col].fillna(median_val)
            print(f"  {col}: Filled with median ({median_val:.2f})")
        else:
            mode_val = merged[col].mode()[0] if len(merged[col].mode()) > 0 else 'Unknown'
            merged[col] = merged[col].fillna(mode_val)
            print(f"  {col}: Filled with mode ({mode_val})")
else:
    print("\nNo remaining nulls! ✓")


STEP 8: HANDLING ANY REMAINING NULLS

Remaining nulls found in 1 columns:
  phase: 1 nulls

Filling remaining nulls with column median (last resort)...
  phase: Filled with mode (Luteal)


In [33]:
# ============================================
# FINAL VERIFICATION
# ============================================

print("\n" + "="*80)
print("IMPUTATION COMPLETE - FINAL VERIFICATION")
print("="*80)

print(f"\nDataset after imputation:")
print(f"  Shape: {merged.shape}")
print(f"  Total missing: {merged.isnull().sum().sum()}")
print(f"  Missing %: {(merged.isnull().sum().sum() / (merged.shape[0] * merged.shape[1]) * 100):.2f}%")

if merged.isnull().sum().sum() == 0:
    print("\n" + "="*80)
    print("SUCCESS! NO MISSING VALUES REMAINING")
    print("="*80)
    print(f"\n✓ Dataset has {len(merged)} rows and {len(merged.columns)} columns")
    print(f"✓ All {merged['id'].nunique()} subjects have complete data")
    print(f"✓ Ready for ML model training!")
else:
    print(f"\nWARNING: Still have {merged.isnull().sum().sum()} missing values")
    print("Check remaining nulls above")

# Show comparison
print(f"\nImputation summary:")
print(f"  Before: {merged_before_imputation.isnull().sum().sum():,} missing cells")
print(f"  After:  {merged.isnull().sum().sum():,} missing cells")
print(f"  Filled: {merged_before_imputation.isnull().sum().sum() - merged.isnull().sum().sum():,} cells")


IMPUTATION COMPLETE - FINAL VERIFICATION

Dataset after imputation:
  Shape: (2970, 61)
  Total missing: 0
  Missing %: 0.00%

SUCCESS! NO MISSING VALUES REMAINING

✓ Dataset has 2970 rows and 61 columns
✓ All 33 subjects have complete data
✓ Ready for ML model training!

Imputation summary:
  Before: 10,735 missing cells
  After:  0 missing cells
  Filled: 10,735 cells


In [34]:
# ============================================
# DATA VALIDATION CHECKS
# ============================================

print("\n" + "="*80)
print("DATA VALIDATION CHECKS")
print("="*80)

# Check 1: No missing values
print("\n1. Missing Values Check:")
missing = merged.isnull().sum().sum()
if missing == 0:
    print("   ✓ PASS: No missing values")
else:
    print(f"   ✗ FAIL: {missing} missing values found!")

# Check 2: No duplicate rows
print("\n2. Duplicate Rows Check:")
duplicates = merged.duplicated(subset=['id', 'day_in_study']).sum()
if duplicates == 0:
    print("   ✓ PASS: No duplicate id/day combinations")
else:
    print(f"   ✗ FAIL: {duplicates} duplicate rows found!")

# Check 3: Target variable exists
print("\n3. Target Variable Check:")
if 'phase' in merged.columns:
    phase_missing = merged['phase'].isnull().sum()
    phase_values = merged['phase'].value_counts()
    print(f"   ✓ PASS: 'phase' column exists")
    print(f"   Missing in phase: {phase_missing}")
    print(f"   Phase distribution:")
    for phase, count in phase_values.items():
        print(f"      {phase}: {count} ({count/len(merged)*100:.1f}%)")
else:
    print("   ✗ FAIL: 'phase' column not found!")

# Check 4: All subjects have data
print("\n4. Subject Coverage Check:")
days_per_subject = merged.groupby('id').size()
print(f"   Total subjects: {merged['id'].nunique()}")
print(f"   Days per subject:")
print(f"      Min: {days_per_subject.min()}")
print(f"      Max: {days_per_subject.max()}")
print(f"      Mean: {days_per_subject.mean():.1f}")
print(f"      Median: {days_per_subject.median():.0f}")

# Check 5: Reasonable value ranges
print("\n5. Value Range Checks:")

checks = [
    ('glucose_mean', 0, 20, 'mmol/L'),
    ('bpm_mean', 40, 200, 'bpm'),
    ('temp_diff_from_baseline_mean', -5, 5, '°C'),
    ('calories_daily', 0, 8000, 'calories'),
    ('steps_daily', 0, 50000, 'steps'),
]

for col, min_val, max_val, unit in checks:
    if col in merged.columns:
        col_min = merged[col].min()
        col_max = merged[col].max()
        if col_min >= min_val and col_max <= max_val:
            print(f"   ✓ {col}: [{col_min:.1f}, {col_max:.1f}] {unit}")
        else:
            print(f"   ⚠ {col}: [{col_min:.1f}, {col_max:.1f}] {unit} (check outliers)")

# Check 6: Data types are correct
print("\n6. Data Type Check:")
print(f"   Numeric columns: {len(merged.select_dtypes(include=['float64', 'int64']).columns)}")
print(f"   Categorical columns: {len(merged.select_dtypes(include=['object']).columns)}")
print(f"   Boolean columns: {len(merged.select_dtypes(include=['bool']).columns)}")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)


DATA VALIDATION CHECKS

1. Missing Values Check:
   ✓ PASS: No missing values

2. Duplicate Rows Check:
   ✓ PASS: No duplicate id/day combinations

3. Target Variable Check:
   ✓ PASS: 'phase' column exists
   Missing in phase: 0
   Phase distribution:
      Luteal: 953 (32.1%)
      Follicular: 805 (27.1%)
      Fertility: 656 (22.1%)
      Menstrual: 556 (18.7%)

4. Subject Coverage Check:
   Total subjects: 33
   Days per subject:
      Min: 90
      Max: 90
      Mean: 90.0
      Median: 90

5. Value Range Checks:
   ⚠ glucose_mean: [3.2, 155.5] mmol/L (check outliers)
   ✓ bpm_mean: [58.7, 131.0] bpm
   ⚠ temp_diff_from_baseline_mean: [-5.4, 2.2] °C (check outliers)
   ✓ calories_daily: [1210.0, 6430.0] calories
   ✓ steps_daily: [0.0, 40675.0] steps

6. Data Type Check:
   Numeric columns: 39
   Categorical columns: 21
   Boolean columns: 1

VALIDATION COMPLETE


### SAVE ###

In [35]:
# ============================================
# SAVE FINAL DATASET
# ============================================

print("="*80)
print("SAVING FINAL DATASET")
print("="*80)

# Save to CSV
output_path = '/home/jupyter/redo/redo_merged_dataset.csv'
merged.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")
print(f"  Rows: {len(merged)}")
print(f"  Columns: {len(merged.columns)}")
print(f"  Size: {merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Also save a backup
backup_path = '/home/jupyter/redo/redo_merged_dataset_backup.csv'
merged.to_csv(backup_path, index=False)
print(f"\n✓ Backup saved to: {backup_path}")

SAVING FINAL DATASET

✓ Saved to: /home/jupyter/redo/redo_merged_dataset.csv
  Rows: 2970
  Columns: 61
  Size: 4.88 MB

✓ Backup saved to: /home/jupyter/redo/redo_merged_dataset_backup.csv


In [36]:
# ============================================
# CREATE DATA DICTIONARY
# ============================================

print("\n" + "="*80)
print("DATA DICTIONARY")
print("="*80)

data_dict = {
    'Column Name': [],
    'Data Type': [],
    'Description': [],
    'Missing (%)': [],
    'Unique Values': []
}

for col in merged.columns:
    data_dict['Column Name'].append(col)
    data_dict['Data Type'].append(str(merged[col].dtype))
    data_dict['Missing (%)'].append(f"{(merged[col].isnull().sum() / len(merged) * 100):.1f}%")
    data_dict['Unique Values'].append(merged[col].nunique())
    
    # Add descriptions
    if col == 'phase':
        desc = "TARGET VARIABLE - Menstrual cycle phase (Follicular/Fertility/Luteal/Menstrual)"
    elif col in ['id', 'day_in_study', 'is_weekend']:
        desc = "Identifier/time column"
    elif 'glucose' in col:
        desc = "Glucose level from CGM (mmol/L)"
    elif 'bpm' in col:
        desc = "Heart rate (beats per minute)"
    elif 'temp' in col:
        desc = "Body temperature measurement"
    elif 'exercise' in col or 'activity' in col:
        desc = "Exercise/activity metric"
    elif 'sleep' in col or 'duration' in col.lower():
        desc = "Sleep measurement"
    elif col in ['lh', 'estrogen']:
        desc = "Hormone level from Mira device"
    elif col in ['flow_volume', 'flow_color']:
        desc = "Menstrual flow characteristic"
    elif col in ['appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 
                 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 
                 'indigestion', 'bloating']:
        desc = "Self-reported symptom (0-5 scale)"
    elif col in ['birth_year', 'gender', 'ethnicity', 'education', 'sexually_active', 
                 'self_report_menstrual_health_literacy', 'age_of_first_menarche']:
        desc = "Subject demographic information"
    else:
        desc = "Physiological measurement"
    
    data_dict['Description'].append(desc)

# Create DataFrame
data_dict_df = pd.DataFrame(data_dict)

# Save to CSV
dict_path = '/home/jupyter/redo/data_dictionary.csv'
data_dict_df.to_csv(dict_path, index=False)
print(f"\n✓ Data dictionary saved to: {dict_path}")
print(f"\nPreview:")
print(data_dict_df.head(10).to_string(index=False))


DATA DICTIONARY

✓ Data dictionary saved to: /home/jupyter/redo/data_dictionary.csv

Preview:
  Column Name Data Type                                                                     Description Missing (%)  Unique Values
           id     int64                                                          Identifier/time column        0.0%             33
   is_weekend      bool                                                          Identifier/time column        0.0%              2
 day_in_study     int64                                                          Identifier/time column        0.0%             90
        phase    object TARGET VARIABLE - Menstrual cycle phase (Follicular/Fertility/Luteal/Menstrual)        0.0%              4
           lh   float64                                                  Hormone level from Mira device        0.0%            250
     estrogen   float64                                                  Hormone level from Mira device        0.0%    

In [37]:
# ============================================
# SUMMARY STATISTICS FOR TEAMMATES
# ============================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Numeric columns only
numeric_cols = merged.select_dtypes(include=['float64', 'int64']).columns
summary_stats = merged[numeric_cols].describe().T

# Add additional stats
summary_stats['missing'] = merged[numeric_cols].isnull().sum()
summary_stats['missing_pct'] = (merged[numeric_cols].isnull().sum() / len(merged) * 100).round(2)

# Save summary
stats_path = '/home/jupyter/redo/summary_statistics.csv'
summary_stats.to_csv(stats_path)
print(f"\n✓ Summary statistics saved to: {stats_path}")
print(f"\nPreview (first 10 columns):")
print(summary_stats.head(10))


SUMMARY STATISTICS

✓ Summary statistics saved to: /home/jupyter/redo/summary_statistics.csv

Preview (first 10 columns):
                 count         mean          std          min          25%  \
id              2970.0    26.121212    14.840528     2.000000    13.000000   
day_in_study    2970.0    45.500000    25.983533     1.000000    23.000000   
lh              2970.0     5.612357     7.407135     0.000000     2.500000   
estrogen        2970.0   143.922525   115.640361     0.000000    67.000000   
calories_daily  2970.0  1982.349832   410.943765  1210.000000  1712.000000   
steps_daily     2970.0  6708.194613  4767.235766     0.000000  3083.250000   
bpm_mean        2970.0    79.591661     8.264622    58.668206    73.458487   
bpm_min         2970.0    53.982828     6.757358    38.000000    50.000000   
bpm_max         2970.0   141.624242    16.557611    86.000000   130.000000   
bpm_std         2970.0    14.356683     3.907762     3.189688    11.692615   

                  

In [25]:
'''

# FINAL CLEANUP - Only run after verifying all aggregations are correct
print("Cleaning up original datasets...")
originals_to_delete = ['heart_rate', 'calories', 'wrist_temperature', 'steps', 
                       'estimated_oxygen_variation', 'glucose', 'respiratory_rate_summary',
                       'sleep_score', 'stress_score', 'computed_temperature', 'sleep', 'exercise']

for name in originals_to_delete:
    if name in datasets:
        del datasets[name]
        
gc.collect()
print("Cleanup complete!")

SyntaxError: incomplete input (3738418730.py, line 1)